# PodcastGen-Agent
Generate full podcast episodes from a topic using AI.

**Runtime**: Select GPU (T4 is fine)

In [ ]:
# Install pinned dependencies (fixes coqui-tts + transformers clash)
!pip install -q --upgrade pip
!pip install -q "transformers==4.43.3" "tokenizers<0.20"
!pip install -q "langgraph>=0.2.0,<0.3.0" "langgraph-checkpoint-sqlite>=2.0.0,<3.0.0"
!pip install -q tenacity bitsandbytes accelerate duckduckgo-search pydub scipy tqdm
!pip install -q "coqui-tts>=0.24.0,<0.26.0"
!pip install -q --force-reinstall --no-deps "transformers==4.43.3"
!pip install -q av
!pip install -q "audiocraft>=1.3.0,<2.0.0" --no-deps
!pip install -q encodec flashy num2words xformers torchmetrics demucs hydra-core hydra-colorlog
!pip install -q --force-reinstall --no-deps "transformers==4.43.3"
!apt-get install -qq ffmpeg

import transformers
try:
    from transformers.pytorch_utils import isin_mps_friendly
except ImportError:
    import torch
    import transformers.pytorch_utils as pytorch_utils
    pytorch_utils.isin_mps_friendly = torch.isin

from TTS.api import TTS
import langgraph, pydub, audiocraft
print("transformers", transformers.__version__)
print("All dependencies installed successfully!")

In [ ]:
import os
if os.path.isdir("PodcastGen-Agent"):
    %cd PodcastGen-Agent
    !git pull
else:
    !git clone https://github.com/Rashal10/PodcastGen-Agent.git
    %cd PodcastGen-Agent

!pip install -q -e .
!pip install -q --force-reinstall --no-deps "transformers==4.43.3"
print("Repo ready")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os

os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["PODCAST_LOG_LEVEL"] = "INFO"

TOPIC = "LoRA and QLoRA"
DURATION = 5

!python -m podcast_gen_agent.main "{TOPIC}" --duration {DURATION}

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Playing: {latest}")
    display(Audio(str(latest)))
else:
    print("No podcast mp3 found. Check the generation cell for errors.")

In [ ]:
from pathlib import Path
from google.colab import files

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Downloading: {latest}")
    files.download(str(latest))
else:
    print("No podcast file found under outputs/. Check the generation cell for errors.")